# Stage 1b — YOLO + ByteTrack (Kaggle, batch video processing)

Takes the fine-tuned detector from Stage 1a and runs ByteTrack over a folder of videos, producing
one `tracks.csv` per video plus a combined file, in the schema Stage 2 (temporal windowing) expects.

**A correction from Stage 1a, applied here.** The confidence sweep in Stage 1a picked
`CONF_THRESHOLD = 0.20` for *standalone detection* precision/recall. That value should **not**
be passed directly as the tracker's only confidence gate — doing so would filter out low-confidence
boxes before ByteTrack ever sees them, defeating its entire second-stage matching design (the
"Byte" step: recovering occluded objects using boxes below the high-confidence threshold).

Instead, this stage passes a **low** confidence to the detector call (`0.10`), and lets a custom
`bytetrack.yaml` do the two-tier logic explicitly:

| Parameter | Value | Role |
|---|---|---|
| `track_high_thresh` | 0.50 | First-stage match: confident boxes matched to predicted track positions |
| `track_low_thresh` | 0.20 (Stage 1a's tuned value) | Second-stage match: recovers occluded/uncertain boxes |
| `new_track_thresh` | 0.60 | Only spawns a **new** track ID from a confident detection — prevents noise from becoming spurious tracks |
| `track_buffer` | 30 frames | How long a lost track stays alive before being dropped |

This is the ByteTrack mechanism as designed, not a workaround.

## 0. Setup

In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.2 MB/s eta 0:00:00


In [2]:
import os, re, glob, json
from pathlib import Path
from collections import defaultdict

import cv2
import yaml
import pandas as pd
from ultralytics import YOLO

print("Setup ready.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Setup ready.


## 1. Load Stage 1a artifacts

Attach the Kaggle Dataset you saved from Stage 1a as a data source, then point these paths at it.

In [3]:
# Paste the exact copied paths inside the quotes
WEIGHTS_PATH = "/kaggle/input/datasets/aryanakhare/directory/yolo11x_surv_v1.pt"
CONFIG_PATH  = "/kaggle/input/datasets/aryanakhare/directory/stage1_config.yaml"

assert os.path.exists(WEIGHTS_PATH), f"Attach Stage 1a's output dataset first. Missing: {WEIGHTS_PATH}"

stage1_cfg = yaml.safe_load(open(CONFIG_PATH))
CLASS_NAMES = {int(k): v for k, v in stage1_cfg["names"].items()}
TUNED_DETECT_CONF = stage1_cfg["conf_threshold"]   

print("Classes:", CLASS_NAMES)
print("Stage 1a tuned confidence:", TUNED_DETECT_CONF)

Classes: {0: 'Person', 1: 'Car', 2: 'Carrying_object', 3: 'Bike', 4: 'Gun', 5: 'Knife'}
Stage 1a tuned confidence: 0.2


## 2. Video and camera ID resolution

In [4]:
import os
from pathlib import Path

# Point this at the one clip you want to run
VIDEO_PATH = "/kaggle/input/datasets/aryanakhare/burglary-test-video/Burglary001_x264A.mp4"

def resolve_video_camera_id(video_path: str):
    """
    Returns (video_id, camera_id) for an mp4 file.
    Since there is no complex folder structure, camera_id = video_id.
    """
    stem = Path(video_path).stem
    return stem, stem

assert os.path.exists(VIDEO_PATH), f"Set VIDEO_PATH to a real video file: {VIDEO_PATH}"

video_id, camera_id = resolve_video_camera_id(VIDEO_PATH)
print(f"File: {Path(VIDEO_PATH).name}")
print(f"  -> Video ID:  {video_id}")
print(f"  -> Camera ID: {camera_id}")


File: Burglary001_x264A.mp4
  -> Video ID:  Burglary001_x264A
  -> Camera ID: Burglary001_x264A


## 3. The tracker module

This class is the reusable "model file" — it gets saved to a standalone `.py` in Section 6
so it can be imported elsewhere (e.g. the testing notebook, or Stage 2).

In [5]:
BYTETRACK_CONFIG = {
    "tracker_type":     "bytetrack",
    "track_high_thresh": 0.50,
    "track_low_thresh":  TUNED_DETECT_CONF,
    "new_track_thresh":  0.60,
    "track_buffer":      30,
    "match_thresh":      0.80,
    "fuse_score":        True,
}

BYTETRACK_YAML_PATH = "/kaggle/working/bytetrack_custom.yaml"
with open(BYTETRACK_YAML_PATH, "w") as f:
    yaml.safe_dump(BYTETRACK_CONFIG, f, sort_keys=False)

print(open(BYTETRACK_YAML_PATH).read())

tracker_type: bytetrack
track_high_thresh: 0.5
track_low_thresh: 0.2
new_track_thresh: 0.6
track_buffer: 30
match_thresh: 0.8
fuse_score: true



In [6]:
class SurveillanceTracker:
    """
    YOLO detection + ByteTrack multi-object tracking for surveillance video.

    Wraps Ultralytics' built-in tracker so detection and tracking run as a single
    pass per video, and normalises output into the schema the rest of the
    pipeline (temporal windowing, event extraction) expects.

    IMPORTANT: call `process_video` once per video. Never reuse `persist=True`
    across different videos in the same tracker instance -- track IDs must not
    bleed across unrelated footage.
    """

    SCHEMA_COLUMNS = [
        "video_id", "camera_id", "frame_id", "timestamp_sec",
        "x1", "y1", "x2", "y2",
        "confidence", "class_id", "class_name", "track_id",
    ]

    def __init__(self, weights_path: str, class_names: dict,
                 bytetrack_yaml: str, detect_conf: float = 0.10,
                 imgsz: int = 640, frame_stride: int = 1):
        """
        detect_conf: passed to the detector call itself. Kept LOW (default 0.10)
            so ByteTrack's second-stage matching has low-confidence boxes to work
            with -- this is not the same as Stage 1a's reporting threshold.
        frame_stride: 1 = process every frame (recommended for tracking quality;
            ByteTrack's Kalman motion model assumes small inter-frame motion).
            Increasing this trades tracking robustness for speed.
        """
        self.model = YOLO(weights_path)
        self.class_names = class_names
        self.bytetrack_yaml = bytetrack_yaml
        self.detect_conf = detect_conf
        self.imgsz = imgsz
        self.frame_stride = frame_stride

    def process_video(self, video_path: str, video_id: str = None,
                       camera_id: str = None, resolver=None) -> pd.DataFrame:
        """Run detection+tracking over one video. Returns a DataFrame in SCHEMA_COLUMNS."""
        if video_id is None or camera_id is None:
            if resolver is not None:
                video_id, camera_id = resolver(video_path)
            else:
                video_id = Path(video_path).stem
                camera_id = video_id

        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        cap.release()

        rows = []
        # stream=True keeps memory bounded over long videos
        results_gen = self.model.track(
            source=video_path,
            tracker=self.bytetrack_yaml,
            conf=self.detect_conf,
            imgsz=self.imgsz,
            vid_stride=self.frame_stride,
            persist=False,     # fresh tracker state for THIS video only
            stream=True,
            verbose=False,
        )

        for frame_idx, result in enumerate(results_gen):
            true_frame = frame_idx * self.frame_stride
            timestamp = true_frame / fps

            if result.boxes is None or result.boxes.id is None:
                continue   # no confirmed tracks this frame

            boxes = result.boxes
            for i in range(len(boxes)):
                cls_id = int(boxes.cls[i])
                rows.append({
                    "video_id":     video_id,
                    "camera_id":    camera_id,
                    "frame_id":     true_frame,
                    "timestamp_sec": round(timestamp, 3),
                    "x1": round(float(boxes.xyxy[i][0]), 1),
                    "y1": round(float(boxes.xyxy[i][1]), 1),
                    "x2": round(float(boxes.xyxy[i][2]), 1),
                    "y2": round(float(boxes.xyxy[i][3]), 1),
                    "confidence": round(float(boxes.conf[i]), 3),
                    "class_id": cls_id,
                    "class_name": self.class_names.get(cls_id, str(cls_id)),
                    "track_id": int(boxes.id[i]),
                })

        return pd.DataFrame(rows, columns=self.SCHEMA_COLUMNS)

    def process_folder(self, folder: str, resolver, pattern: str = "*.mp4") -> pd.DataFrame:
        """Process every video in a folder. Returns the combined DataFrame."""
        video_paths = sorted(glob.glob(os.path.join(folder, pattern)))
        all_dfs = []
        for vp in video_paths:
            video_id, camera_id = resolver(vp)
            print(f"Processing {video_id} (camera: {camera_id}) ...")
            df = self.process_video(vp, video_id=video_id, camera_id=camera_id)
            print(f"  -> {len(df)} track rows, {df['track_id'].nunique() if len(df) else 0} unique tracks")
            all_dfs.append(df)
        if not all_dfs:
            return pd.DataFrame(columns=self.SCHEMA_COLUMNS)
        return pd.concat(all_dfs, ignore_index=True)


print("SurveillanceTracker defined.")

SurveillanceTracker defined.


## 4. Run on a single video

Point `VIDEO_PATH` at one clip. This still uses the same `SurveillanceTracker` class from Section 3 unchanged -- just calling `process_video` directly instead of `process_folder`, so nothing about the tracker logic itself changes.


In [7]:
OUTPUT_DIR = "/kaggle/working/tracks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

tracker = SurveillanceTracker(
    weights_path=WEIGHTS_PATH,
    class_names=CLASS_NAMES,
    bytetrack_yaml=BYTETRACK_YAML_PATH,
    detect_conf=0.10,
    frame_stride=1,   # every frame -- see note in class docstring
)

print(f"Processing {video_id} (camera: {camera_id}) ...")
combined = tracker.process_video(VIDEO_PATH, video_id=video_id, camera_id=camera_id)
print(f"  -> {len(combined)} track rows, "
      f"{combined['track_id'].nunique() if len(combined) else 0} unique tracks")


Processing Burglary001_x264A (camera: Burglary001_x264A) ...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 186ms
 Downloaded lap
Prepared 1 package in 33ms
Installed 1 package in 4ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

  -> 334 track rows, 11 unique tracks


In [8]:
# Single video, so just save one CSV -- no groupby needed.
tracks_path = f"{OUTPUT_DIR}/{video_id}_tracks.csv"
combined.to_csv(tracks_path, index=False)
print(f"Saved tracks: {tracks_path}")

# Also write the "combined" file under the name Stage 2 expects, so the rest
# of the pipeline (and the artifacts-to-carry-forward check in Section 7) works
# unchanged whether you ran one video or a folder.
combined_path = "/kaggle/working/tracks_all.csv"
combined.to_csv(combined_path, index=False)
print(f"Saved combined tracks: {combined_path}")
combined.head(10)


Saved tracks: /kaggle/working/tracks/Burglary001_x264A_tracks.csv
Saved combined tracks: /kaggle/working/tracks_all.csv


,video_id,camera_id,frame_id,timestamp_sec,x1,y1,x2,y2,confidence,class_id,class_name,track_id
0,Burglary001_x264A,Burglary001_x264A,764,25.467,154.5,38.8,213.2,169.3,0.816,0,Person,2
1,Burglary001_x264A,Burglary001_x264A,765,25.500,149.6,31.1,212.3,170.4,0.557,0,Person,2
2,Burglary001_x264A,Burglary001_x264A,766,25.533,151.1,27.5,215.0,170.2,0.671,0,Person,2
3,Burglary001_x264A,Burglary001_x264A,767,25.567,150.9,27.8,214.5,170.4,0.782,0,Person,2
4,Burglary001_x264A,Burglary001_x264A,768,25.600,151.0,25.1,215.0,169.6,0.835,0,Person,2
5,Burglary001_x264A,Burglary001_x264A,769,25.633,147.6,31.7,209.2,170.0,0.646,0,Person,2
6,Burglary001_x264A,Burglary001_x264A,770,25.667,144.3,29.9,207.3,170.5,0.473,0,Person,2
7,Burglary001_x264A,Burglary001_x264A,771,25.700,143.1,28.9,206.8,170.6,0.796,0,Person,2
8,Burglary001_x264A,Burglary001_x264A,772,25.733,141.5,27.3,206.6,171.6,0.810,0,Person,2
9,Burglary001_x264A,Burglary001_x264A,773,25.767,140.6,29.4,206.5,175.1,0.684,0,Person,2


## 5. Track-quality checkpoint

Before trusting this output, check for two failure signs: tracks that are implausibly short
(ID churn — the same object keeps getting a new ID), and videos with zero tracks (detector or
tracker failing silently on that clip).

In [9]:
def track_quality_report(df: pd.DataFrame, fps_assumed: float = 25.0):
    if df.empty:
        print("No tracks at all -- check detect_conf, weights path, and video files.")
        return

    per_track = df.groupby(["video_id", "track_id"]).agg(
        n_frames=("frame_id", "count"),
        class_name=("class_name", "first"),
    ).reset_index()

    per_track["duration_sec"] = per_track["n_frames"] / fps_assumed

    print(f"Total tracks: {len(per_track)}")
    print(f"Median track duration: {per_track['duration_sec'].median():.2f}s")
    print(f"Tracks shorter than 0.5s (possible ID churn): "
          f"{(per_track['duration_sec'] < 0.5).sum()} "
          f"({(per_track['duration_sec'] < 0.5).mean()*100:.1f}%)")

    print("\nPer-video track counts:")
    print(df.groupby("video_id")["track_id"].nunique())

    print("\nPer-class track counts:")
    print(per_track.groupby("class_name")["track_id"].count())

    return per_track

quality = track_quality_report(combined)

Total tracks: 11
Median track duration: 0.96s
Tracks shorter than 0.5s (possible ID churn): 5 (45.5%)

Per-video track counts:
video_id
Burglary001_x264A    11
Name: track_id, dtype: int64

Per-class track counts:
class_name
Gun       4
Person    7
Name: track_id, dtype: int64


**What to look for:** a high share of sub-0.5-second tracks usually means the detector's
recall gap (Person recall was 0.604 before the Stage 1a fixes) is still causing missed frames,
which ByteTrack reads as the object disappearing and reappearing as a new ID. If this share is
high, that's a signal to revisit Stage 1a's gate check before trusting Stage 1b's output.

## 6. Save the reusable module file

In [10]:
MODULE_SOURCE = '''"""
SurveillanceTracker -- YOLO + ByteTrack for surveillance video.

Generated from Stage 1b. Import this module wherever the pipeline needs
detection+tracking (Stage 2 temporal windowing, the testing notebook, etc.).
"""

import os
import glob
import re
from pathlib import Path

import cv2
import pandas as pd
from ultralytics import YOLO


def resolve_video_camera_id(video_path: str):
    """Returns (video_id, camera_id). See Stage 1b notebook Section 2 for details
    and the caveat about verifying the VIRAT scene-code regex against real filenames."""
    stem = Path(video_path).stem
    video_id = stem
    m = re.match(r"VIRAT_S_(\\d{2})(\\d{4})", stem)
    if m:
        return video_id, f"scene_{m.group(1)}"
    return video_id, video_id


class SurveillanceTracker:
    """YOLO detection + ByteTrack multi-object tracking for surveillance video."""

    SCHEMA_COLUMNS = [
        "video_id", "camera_id", "frame_id", "timestamp_sec",
        "x1", "y1", "x2", "y2",
        "confidence", "class_id", "class_name", "track_id",
    ]

    def __init__(self, weights_path, class_names, bytetrack_yaml,
                 detect_conf=0.10, imgsz=640, frame_stride=1):
        self.model = YOLO(weights_path)
        self.class_names = class_names
        self.bytetrack_yaml = bytetrack_yaml
        self.detect_conf = detect_conf
        self.imgsz = imgsz
        self.frame_stride = frame_stride

    def process_video(self, video_path, video_id=None, camera_id=None, resolver=None):
        if video_id is None or camera_id is None:
            if resolver is not None:
                video_id, camera_id = resolver(video_path)
            else:
                video_id = Path(video_path).stem
                camera_id = video_id

        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        cap.release()

        rows = []
        results_gen = self.model.track(
            source=video_path, tracker=self.bytetrack_yaml,
            conf=self.detect_conf, imgsz=self.imgsz,
            vid_stride=self.frame_stride, persist=False,
            stream=True, verbose=False,
        )

        for frame_idx, result in enumerate(results_gen):
            true_frame = frame_idx * self.frame_stride
            timestamp = true_frame / fps
            if result.boxes is None or result.boxes.id is None:
                continue
            boxes = result.boxes
            for i in range(len(boxes)):
                cls_id = int(boxes.cls[i])
                rows.append({
                    "video_id": video_id, "camera_id": camera_id,
                    "frame_id": true_frame, "timestamp_sec": round(timestamp, 3),
                    "x1": round(float(boxes.xyxy[i][0]), 1),
                    "y1": round(float(boxes.xyxy[i][1]), 1),
                    "x2": round(float(boxes.xyxy[i][2]), 1),
                    "y2": round(float(boxes.xyxy[i][3]), 1),
                    "confidence": round(float(boxes.conf[i]), 3),
                    "class_id": cls_id,
                    "class_name": self.class_names.get(cls_id, str(cls_id)),
                    "track_id": int(boxes.id[i]),
                })
        return pd.DataFrame(rows, columns=self.SCHEMA_COLUMNS)

    def process_folder(self, folder, resolver, pattern="*.mp4"):
        video_paths = sorted(glob.glob(os.path.join(folder, pattern)))
        all_dfs = []
        for vp in video_paths:
            video_id, camera_id = resolver(vp)
            df = self.process_video(vp, video_id=video_id, camera_id=camera_id)
            all_dfs.append(df)
        if not all_dfs:
            return pd.DataFrame(columns=self.SCHEMA_COLUMNS)
        return pd.concat(all_dfs, ignore_index=True)
'''

with open("/kaggle/working/surveillance_tracker.py", "w") as f:
    f.write(MODULE_SOURCE)

print("Saved: /kaggle/working/surveillance_tracker.py")
print(f"({len(MODULE_SOURCE.splitlines())} lines)")

Saved: /kaggle/working/surveillance_tracker.py
(98 lines)


## 7. Artifacts to carry forward

Save these as a Kaggle Dataset for the testing notebook and for Stage 2 (temporal windowing):

- `surveillance_tracker.py` — the importable module
- `bytetrack_custom.yaml` — the tuned tracker config
- `tracks_all.csv` — combined tracking output (or per-video CSVs in `tracks/`)

In [11]:
for f in ["/kaggle/working/surveillance_tracker.py",
          "/kaggle/working/bytetrack_custom.yaml",
          "/kaggle/working/tracks_all.csv"]:
    exists = os.path.exists(f)
    size = f"{os.path.getsize(f)/1024:.1f} KB" if exists else "MISSING"
    print(f"{'OK ' if exists else '!! '}{f}  ({size})")

OK /kaggle/working/surveillance_tracker.py  (3.6 KB)
OK /kaggle/working/bytetrack_custom.yaml  (0.1 KB)
OK /kaggle/working/tracks_all.csv  (28.1 KB)
